# TP - Store Catalog (recursion over JSON you did not design)

**Practical exercise - not a LeetCode problem.**

`../data/catalog.json` is a store's catalog. A category can hold **sub-categories**,
**products**, or **both**, nested as deep as the store feels like. That is a
**tree** - except this time *you did not build it*, and it is not made of your
`TreeNode` class. It is plain `dict`s and `list`s that arrived from a file.

The whole point: your `#104` depth and your TP org-chart walks were recursion
over a tree you constructed. Real data is a tree someone *else* shaped, handed to
you as JSON. The algorithms are the same. Learning to see the tree inside the
JSON is the skill.

The shape of one node:

```json
{
  "name": "Laptops",
  "categories": [ ... more category nodes ... ],   // optional
  "products":   [ {"name": ..., "price": ..., "stock": ...}, ... ]   // optional
}
```

A category may be missing `"categories"`, or missing `"products"`, or have both.
`dict.get("categories", [])` is your friend - it hands back an empty list when
the key is absent, so your loops just do nothing instead of crashing.


### Exercise 0 - load it

Read the file and look at the top level. Same relative path as the org-chart TP:
you are in `tp/`, the data is in `../data/`.

In [ ]:
import json

with open("../data/catalog.json", encoding="utf-8") as f:
    catalog = json.load(f)

print(catalog["name"])
print("top-level categories:", [c["name"] for c in catalog["categories"]])


### Exercise 1 - count every product

How many products are in the whole store, at every depth?

A category's product count is: *its own products*, **plus** the product count of
each of its sub-categories. That second half is the recursion - the same
`sum(...)` over children you wrote for the org chart's headcount.

- Base-ish case: a category with no sub-categories just returns `len(its products)`.
- What does `.get("products", [])` give you for a category that has none?

**Expected total: 11.**

In [ ]:
def count_products(node) -> int:
    # TODO : own products + products in every sub-category
    pass


# print(count_products(catalog))   # 11


### Exercise 2 - total inventory value

The money sitting on the shelves: for every product, `price * stock`, summed
across the entire tree. Same recursion shape as Exercise 1 - only the thing you
add up changes. (Notice: once you have the *shape* of one of these walks, the
rest are copy-and-tweak. That is the pattern paying off.)

**Expected total: 57,985.**

In [ ]:
def total_value(node) -> int:
    # TODO : sum(price * stock) over every product, at every depth
    pass


# print(total_value(catalog))   # 57985


### Exercise 3 - how deep does the catalog go?

The maximum nesting depth of categories. This is `#104 maxDepth` again - but a
node here has a **list** of children, not just left/right. So instead of
`max(left, right)` you need the max over *all* sub-categories.

Count `TechMart` (the root) as depth 1. The `Gaming` branch is the deepest:
`TechMart -> Computers -> Laptops -> Gaming`.

- What is the depth of a category with no sub-categories?
- `max(...)` of an **empty** sequence raises - how do you guard the leaf case?
  (Hint: `max(seq, default=0)` exists.)

**Expected depth: 4.**

In [ ]:
def depth(node) -> int:
    # TODO : 1 + max(depth of each sub-category), or 1 at a leaf category
    pass


# print(depth(catalog))   # 4


### Exercise 4 - find a product by name

Search the whole tree for a product and return it (its `dict`), or `None` if no
product has that name. This is depth-first search - walk into sub-categories,
and the moment you find it, stop and hand it back up.

- When a recursive call finds it, how does the answer travel back up through all
  the callers? (You must *return* the found value, not just discover it and keep
  looping.)

Test with `"AirLite 14"` (should return its dict) and `"Nonexistent"` (`None`).

In [ ]:
def find_product(node, target: str):
    # TODO : return the product dict whose name == target, else None
    pass


# print(find_product(catalog, "AirLite 14"))   # {'name': 'AirLite 14', 'price': 1299, 'stock': 5}
# print(find_product(catalog, "Nonexistent"))  # None


### Exercise 5 - the path to a product (breadcrumbs)

Return the list of category names leading down to a product - its "breadcrumb"
trail. For `"RaptorBook 15"` that is:

```
["TechMart", "Computers", "Laptops", "Gaming"]
```

This is the org chart's **chain of command** in disguise: as you recurse down,
carry the trail so far; when you find the product, the trail *is* the answer.
Think about whether you build the path on the way **down** (pass it in) or on the
way **back up** (prepend as the answer returns). Either works - pick one and know
why.

Return `None` (or `[]`) if the product isn't there.

In [ ]:
def path_to(node, target: str, trail=None):
    # TODO : list of category names from the root down to the product, else None
    pass


# print(path_to(catalog, "RaptorBook 15"))
# ["TechMart", "Computers", "Laptops", "Gaming"]


### Exercise 6 - flatten to a price list

Produce a flat list of `(product_name, price)` for the whole store, in the order
you meet them walking the tree. The tree structure is gone; you are *serializing*
it. This is the inverse of what you did loading the JSON - and exactly what a
"export to CSV" button does under the hood.

**Expected length: 11** (same as Exercise 1 - every product, once).

In [ ]:
def price_list(node, out=None):
    # TODO : collect (name, price) for every product, at every depth
    pass


# rows = price_list(catalog)
# for name, price in rows:
#     print(f"{name:<16} {price}")
# print("count:", len(rows))   # 11


### When you have all six

Look back at what changed between Exercises 1, 2, and 6. The *walk* is identical
every time - descend into every sub-category, touch every product. Only the thing
you **do** at each product differs (count it / add its value / collect its name).
That separation - "traverse" vs "what to do at each node" - is the seed of the
**visitor pattern**, and it is exactly the reusable-`getList`-vs-per-level-work
idea from your zigzag queue, one level up. Same lesson keeps coming back.